In [ ]:
!pip install -q datasets kagglehub pandas

## Load a dataset (`MongoDB/whatscooking.restaurants`) from HuggingFace

In [ ]:
from datasets import load_dataset

hf_dataset = load_dataset("MongoDB/whatscooking.restaurants")

In [ ]:
# Save the dataset splits as CSV files.
for split_name, split_rows in hf_dataset.items():
    hf_df = split_rows.to_pandas()
    hf_df.to_csv(f"/content/mongodb_restaurants_{split_name}.csv", index=False)

## Load a dataset (`team-ai/spam-text-message-classification`) from Kaggle

In [ ]:
import kagglehub

kaggle_df = kagglehub.dataset_load(kagglehub.KaggleDatasetAdapter.PANDAS, "team-ai/spam-text-message-classification", "SPAM text message 20170820 - Data.csv")

## Convert the dataset to JSONL for fine-tuning

The [chat completions](https://developers.openai.com/api/docs/guides/supervised-fine-tuning#build-your-dataset) format is used.

Example:
```json
{
    "messages": [
        { "role": "user", "content": "What is the capital of France?" },
        { "role": "assistant", "content": "The capital of France is Paris." }
    ]
}
```

In [ ]:
SYSTEM_MESSAGE = "Classify the message strictly as 'spam' or 'ham'. Return only one word."
fine_tuning_df = kaggle_df.apply(
    lambda x: [
        { "role": "developer", "content": SYSTEM_MESSAGE },
        { "role": "user", "content": f"Classify the message: \"{x["Message"]}\"" },
        { "role": "assistant", "content": x["Category"] }
    ],
    axis=1
)

fine_tuning_df = fine_tuning_df.to_frame(name="messages")

In [ ]:
fine_tuning_df.to_json("/content/fine_tuning.jsonl", index=False, orient="records", lines=True)